In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# Setup headless browser
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
driver = webdriver.Chrome(options=options)

# Keyword query in URL is already filtered as "water supply"
base_url = "https://projects.worldbank.org/en/projects-operations/projects-list?os={page}&qterm=water%20supply"

# Final data storage
all_projects = []

# Loop through pages (adjust range as needed)
for page in range(0, 100, 20):  # Pages go in steps of 20 (os=0,20,40,...)
    print(f"🔍 Processing page {page}...")
    url = base_url.format(page=page)
    driver.get(url)

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a.project-list-link"))
        )
    except:
        print("⚠️ Timeout on page", page)
        continue

    rows = driver.find_elements(By.XPATH, "//tr[.//a[contains(@class,'project-list-link')]]")

    for row in rows:
        try:
            title = row.find_element(By.CSS_SELECTOR, "a.project-list-link").text.strip()
        except:
            continue

        def safe_extract(field):
            try:
                return row.find_element(By.XPATH, f".//td[@data-th='{field}']").text.strip()
            except:
                return ""

        project_data = {
            "Title": title,
            "Country": safe_extract("Country:"),
            "Project ID": safe_extract("Project ID:"),
            "Commitment Amount": safe_extract("Commitment Amount:"),
            "Status": safe_extract("Status:"),
            "Approval Date": safe_extract("Approval Date:"),
            "Last Updated Date": safe_extract("Last updated Date:"),
            "Last Stage Reached": safe_extract("Last Stage Reached:")
        }

        all_projects.append(project_data)

    # Wait between pages to avoid server blocking
    time.sleep(2)

driver.quit()

# Save to Excel
df = pd.DataFrame(all_projects)
df.to_excel("WorldBank_WaterSupply_Projects1.xlsx", index=False)
print("✅ All data saved to 'WorldBank_WaterSupply_Projects1.xlsx'")


🔍 Processing page 0...
🔍 Processing page 20...
🔍 Processing page 40...
🔍 Processing page 60...
🔍 Processing page 80...
✅ All data saved to 'WorldBank_WaterSupply_Projects1.xlsx'


In [4]:
import requests
import pandas as pd

# Search term and base API URL
search_term = "water supply"
base_url = "https://search.worldbank.org/api/v2/projects"

# Storage
all_projects = []

# Loop through pages (50 rows per page)
for offset in range(0, 200, 50):  # You can increase to get more pages
    params = {
        "qterm": search_term,
        "format": "json",
        "rows": 50,
        "os": offset
    }

    print(f"🔍 Fetching projects at offset {offset}...")
    response = requests.get(base_url, params=params)
    data = response.json()

    # Extract project dictionary
    projects = data.get("projects", {})

    for pid, project in projects.items():
        project_info = {
            "Project ID": project.get("id", ""),
            "Title": project.get("project_name", ""),
            "Country": project.get("countryshortname", ""),
            "Status": project.get("status", ""),
            "Approval Date": project.get("boardapprovaldate", ""),
            "Commitment Amount": project.get("lendprojectcost", ""),
            "Region": project.get("regionname", ""),
            "Sector": project.get("sector", ""),
            "Theme": project.get("theme", "")
        }
        all_projects.append(project_info)

# Convert to DataFrame
df = pd.DataFrame(all_projects)

# Save to Excel
df.to_excel("Water_Supply_Projects_WorldBank.xlsx", index=False)
print("✅ Done! Saved to 'Water_Supply_Projects_WorldBank.xlsx'")


🔍 Fetching projects at offset 0...
🔍 Fetching projects at offset 50...
🔍 Fetching projects at offset 100...
🔍 Fetching projects at offset 150...
✅ Done! Saved to 'Water_Supply_Projects_WorldBank.xlsx'
